# Lab 8 — Adding attention to the digit reader

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kleinric/cv-labs/blob/main/lab-08.ipynb)

**COMS4036A / COMS7050A Computer Vision**

Groups of up to three; one member submits the group's notebook (`.ipynb`) on Moodle. Every member must be able to explain every cell.

Companion reading is [Chapter 8 of the course book](https://courses.ms.wits.ac.za/~richard/cv/book/chapters/08-attention-transformers.html). The digit reader from Lab 7 starts its decoder from one summary of the strip, $s_0$, and never looks at the image again. In this lab you give the reader a state for every column of the CNN's feature map, build two readers that differ mainly in whether the decoder can return to those states, and compare them on strips of six to eight digits and on strips longer than any they were trained on. After twelve epochs on strips of one to eight digits, the reference fixed-context reader reads about 77% of the test strips of six to eight digits exactly and the attention reader about 96%; three reference runs, with seeds 0, 1 and 2, gave 0.763, 0.771 and 0.783 for the first and 0.969, 0.961 and 0.962 for the second. Your run will differ in detail, and the behaviour on strips longer than the training lengths varies from run to run more than these numbers do.

This lab's rung of the skills ladder is **a controlled comparison: change as little as possible, and say what else changed**. The two readers you build use the same encoder design, data, training recipe and seeds, and differ mainly in whether the decoder can return to the column states. Adding attention also adds parameters: the attention reader is about 16% larger, because of its projections and its wider decoder input and output layers. The comparison therefore tests this change to the architecture without separating attention from the extra capacity, and you report both parameter counts so that a reader can take that into account. The same care applies to the attention maps. A weight is a coefficient in the mixture that forms a context; it does not by itself show which part of the image a prediction depended on. Section 5 tests that by replacing a digit and measuring how the readings change, rather than by looking at a map.

## 0. GPU, W&B, and the data

Open this notebook in [Colab](https://colab.research.google.com) with the badge above, or [download it from the book](https://courses.ms.wits.ac.za/~richard/cv/book/labs/lab-08.ipynb) and use **File ▸ Upload notebook**; then **File ▸ Save a copy in Drive**. Set **Runtime ▸ Change runtime type** to a GPU, and log in to W&B; this lab logs to a project called `cv-lab8`.

In [ ]:
# Group members — fill in before submitting.
MEMBERS = [
    # ("Student name", "Student number"),
]
for name, number in MEMBERS:
    print(f"{number}  {name}")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import wandb
wandb.login()

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets

DEV = "cuda" if torch.cuda.is_available() else "cpu"


def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


torch.backends.cudnn.benchmark = False      # repeatable runs on the same GPU
torch.backends.cudnn.deterministic = True
seed_everything(0)
print("device:", DEV)

The digits are MNIST, as in Lab 7: 60,000 training and 10,000 test images of single handwritten digits, each $28 \times 28$ pixels (LeCun, Cortes and Burges). Training strips are built only from the training digits and test strips only from the test digits, so no handwritten digit in a test strip has been seen in training.

In [ ]:
mnist_train = datasets.MNIST("data", train=True, download=True)
mnist_test = datasets.MNIST("data", train=False, download=True)
train_imgs, train_labels = mnist_train.data.numpy(), mnist_train.targets.numpy()
test_imgs, test_labels = mnist_test.data.numpy(), mnist_test.targets.numpy()
print(f"MNIST digits: {len(train_imgs)} train, {len(test_imgs)} test")

The vocabulary and the scoring functions are Lab 7's. The digits 0 to 9 are their own indices, and 10, 11 and 12 are `<sos>`, `<eos>` and `<pad>`. `clean` turns a token sequence into a list of digits by stopping at the first `<eos>` and dropping `<sos>` and `<pad>`. **Exact-string accuracy** is the share of strips whose cleaned prediction equals the target. **Character error rate** (CER) is the number of digit substitutions, deletions and insertions needed to turn the prediction into the target, summed over the strips and divided by the total number of target digits. `by_length` reports both measures separately for each target length.

In [ ]:
SOS, EOS, PAD = 10, 11, 12
VOCAB = 13
SYMBOLS = [str(d) for d in range(10)] + ["<sos>", "<eos>", "<pad>"]


def show_tokens(seq):
    return " ".join(SYMBOLS[int(t)] for t in seq)


def clean(seq):
    """Digits only: stop at the first <eos>, drop <sos> and <pad>."""
    out = []
    for t in seq:
        t = int(t)
        if t == EOS:
            break
        if t < 10:
            out.append(t)
    return out


def edit_distance(a, b):
    """Minimum substitutions, deletions and insertions turning a into b."""
    prev = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        cur = [i] + [0] * len(b)
        for j in range(1, len(b) + 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1,
                         prev[j - 1] + (a[i - 1] != b[j - 1]))
        prev = cur
    return prev[-1]


def exact_string_accuracy(preds, targets):
    """Fraction of strings whose cleaned prediction equals the cleaned target."""
    return float(np.mean([clean(p) == clean(t) for p, t in zip(preds, targets)]))


def cer(preds, targets):
    """Character error rate: total edits over total target digits."""
    edits = sum(edit_distance(clean(p), clean(t)) for p, t in zip(preds, targets))
    return edits / sum(len(clean(t)) for t in targets)


def by_length(preds, targets):
    """{length: (exact-string accuracy, CER, number of strips)}, by target length."""
    rows = {}
    for p, t in zip(preds, targets):
        rows.setdefault(len(clean(t)), []).append((p, t))
    return {L: (exact_string_accuracy(*zip(*r)), cer(*zip(*r)), len(r))
            for L, r in sorted(rows.items())}

## 1. Strips on a wider canvas

The generator is Lab 7's, with the width of the canvas as a parameter. Lab 7's canvas was 224 pixels wide, enough for eight digits. Here it is 336 pixels wide, so that a strip of twelve digits always fits: twelve 28-pixel boxes with no overlap take exactly 336 pixels. As before, neighbouring boxes overlap by between 0 and 6 pixels and the whole string starts at a random horizontal position. The generator now also keeps, for every strip, the index of each digit's source image (`dataset.src[i]`) and the left edge of each digit's box (`dataset.xs[i]`), so that Section 5 can compare the attention weights with where the digits are, and redraw a strip with one digit changed using `draw_strip`.

In [ ]:
H, W = 28, 336
GAP_RANGE = (-6, 0)


def draw_strip(images, idx, xs, width=W):
    """Draw the digits images[idx[k]] with their left edges at xs[k]."""
    canvas = np.zeros((H, width), dtype=np.uint8)
    for i, x in zip(idx, xs):
        np.maximum(canvas[:, x:x + 28], images[i], out=canvas[:, x:x + 28])
    return canvas


def make_strip(images, labels, length, rng, width=W):
    """One strip of `length` digits on a blank canvas 28 pixels high and `width` wide.

    Lab 7's generator, with the canvas width as a parameter. Returns the canvas
    (uint8), the digits, the indices of the source images and the left edge of
    each digit's 28-pixel box.
    """
    idx = rng.integers(0, len(images), size=length)
    gaps = rng.integers(GAP_RANGE[0], GAP_RANGE[1] + 1, size=length - 1)
    span = 28 * length + int(gaps.sum())
    assert span <= width, f"{length} digits do not fit on a canvas {width} wide"
    x0 = int(rng.integers(0, width - span + 1))
    xs = x0 + np.concatenate([[0], np.cumsum(28 + gaps)]).astype(int)
    return draw_strip(images, idx, xs, width), [int(labels[i]) for i in idx], idx, xs


class StripDataset(Dataset):
    """`n` strips, equally many of each length in `lengths`, fixed by `seed`.

    Item i is (image, digits): a float tensor of shape (1, 28, width) in [0, 1]
    and a long tensor holding the digits only. The source-image indices and the
    box left edges of strip i are in self.src[i] and self.xs[i].
    """

    def __init__(self, images, labels, n, lengths, seed, width=W):
        rng = np.random.default_rng(seed)
        lens = np.resize(np.asarray(lengths), n)     # equal counts per length
        rng.shuffle(lens)
        self.images = np.zeros((n, H, width), dtype=np.uint8)
        self.digits, self.src, self.xs = [], [], []
        for j, L in enumerate(lens):
            self.images[j], d, idx, xs = make_strip(images, labels, int(L), rng, width)
            self.digits.append(d)
            self.src.append(idx)
            self.xs.append(xs)

    def __len__(self):
        return len(self.digits)

    def __getitem__(self, i):
        img = torch.from_numpy(self.images[i]).float().div(255).unsqueeze(0)
        return img, torch.tensor(self.digits[i], dtype=torch.long)


test_set = StripDataset(test_imgs, test_labels, 8_000, range(1, 9), seed=1234)
long_set = StripDataset(test_imgs, test_labels, 4_000, range(9, 13), seed=4321)
print(len(test_set), len(long_set))

`test_set` holds 1,000 strips of each length from one to eight, and `long_set` 1,000 of each length from nine to twelve, all from MNIST test digits and identical on every run. Both readers train on strips of one to eight digits, so neither trains on strips of nine to twelve digits; Section 6 uses `long_set`.

The batching function and the masked loss are the ones you wrote in Lab 7. `collate` pads the decoder inputs (`<sos>` followed by the digits) and the targets (the digits followed by `<eos>`) with `<pad>` to a common length, and `masked_loss` averages the cross-entropy over every target position that is not `<pad>`. The CNN is Lab 7's too: four blocks, each a $3 \times 3$ convolution, batch norm, ReLU and a $2 \times 2$ max-pool, at 32, 64, 128 and 128 channels.

In [ ]:
def collate(batch):
    """Stack the images; pad decoder inputs and targets with PAD."""
    images = torch.stack([img for img, _ in batch])
    sos, eos = torch.tensor([SOS]), torch.tensor([EOS])
    dec_in = pad_sequence([torch.cat([sos, d]) for _, d in batch],
                          batch_first=True, padding_value=PAD)
    tgt = pad_sequence([torch.cat([d, eos]) for _, d in batch],
                       batch_first=True, padding_value=PAD)
    return images, dec_in, tgt


def masked_loss(logits, tgt):
    """Mean cross-entropy over the target positions that are not <pad>."""
    return F.cross_entropy(logits.reshape(-1, VOCAB), tgt.reshape(-1), ignore_index=PAD)


def conv_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
                         nn.BatchNorm2d(c_out), nn.ReLU(), nn.MaxPool2d(2))


def make_cnn():
    """Lab 7's CNN: four blocks at 32, 64, 128 and 128 channels."""
    return nn.Sequential(conv_block(1, 32), conv_block(32, 64),
                         conv_block(64, 128), conv_block(128, 128))

**Q1.1.** Display six strips from `test_set` and four from `long_set`, each titled with its digits and with each digit's box outlined from `dataset.xs`.

Before running the CNN, work out three numbers: the shape of the CNN's feature map $F$ for one strip; the distance in pixels between the strip positions that neighbouring columns of $F$ correspond to; and how many columns a 28-pixel digit box spans at that spacing. Check the shape with `make_cnn()`.

Finally, find the window of input pixels that one column of $F$ depends on, its *receptive field*: compute it from the layers by hand, and check it by back-propagating from one column of $F$ of an untrained CNN in evaluation mode, given a random image (`torch.rand`) rather than a blank one, and finding which pixel columns of the input receive a non-zero gradient. Why would a blank image not work for this check? How many digit boxes can that window overlap?

In [ ]:
# YOUR CODE HERE

*Answer:*

## 2. A state for every column

The CNN turns the strip into 21 columns of 128 numbers each. In Lab 7 the decoder's first state was a projection of all of them at once, and the decoder never saw them again. Attention needs the opposite: a collection of states, one per column, that the decoder can return to at every step.

A column of $F$ describes only its own window of the strip. A **bidirectional GRU** makes each column's state describe that column in the context of the whole strip. One GRU reads the 21 columns from left to right and another reads them from right to left, and the state for column $j$ joins the two:

$$
h_j=\bigl[\overrightarrow{h}_j;\,\overleftarrow{h}_j\bigr],\qquad j=1,\ldots,21.
$$

The forward state $\overrightarrow{h}_j$ has read columns 1 to $j$, and the backward state $\overleftarrow{h}_j$ has read columns 21 down to $j$. With 64 units in each direction, each $h_j$ has 128 numbers. The decoder's first state comes from the two final states, the forward one after column 21 and the backward one after column 1,

$$
s_0=\tanh\!\left(W_{\text{init}}\bigl[\overrightarrow{h}_{21};\,\overleftarrow{h}_1\bigr]+b_{\text{init}}\right),
$$

with a width of 64. Both readers in Section 3 use this encoder, so they start from the same kind of $s_0$ and differ in what the decoder can read after that.

`nn.GRU(..., bidirectional=True, batch_first=True)` does both directions in one call. Its first output has shape $(B, 21, 128)$, with the forward states in the first 64 numbers of each column and the backward states in the last 64. Its second output has shape $(2, B, 64)$: index 0 is the forward GRU's final state and index 1 the backward GRU's.

**Q2.1.** Implement `ColumnEncoder(hidden=64, units=64)` with three parts named `cnn` (from `make_cnn()`), `bigru` (an `nn.GRU` from 128 inputs to `units` units in each direction) and `init` (an `nn.Linear(2 * units, hidden)`). Its `forward(images, trace=False)` turns $F$ of shape $(B, 128, 1, 21)$ into a sequence of 21 column vectors, runs the bidirectional GRU over them, and returns two tensors: `memory`, the column states $h_1, \ldots, h_{21}$ as $(B, 21, 128)$, and $s_0$ as $(B, 64)$. When `trace` is true, print the shapes of $F$, the column sequence, `memory`, the final states and $s_0$.

In [ ]:
class ColumnEncoder(nn.Module):
    # YOUR CODE HERE
    ...

**Q2.2.** Before you run it, compute by hand the number of parameters in the bidirectional GRU and in $W_{\text{init}}$ and $b_{\text{init}}$. A GRU layer with $n$ inputs and $u$ units holds three sets of input weights ($u \times n$), recurrent weights ($u \times u$) and two biases ($u$ each). Then call `seed_everything(0)`, build a `ColumnEncoder`, run the trace on a batch of four strips from `test_set`, and print the parameter count of each part.

Then test the claim that a column state describes the whole strip. Take the first eight-digit strip in `test_set`, and draw a second version of it with its last digit erased (`draw_strip` with the last entry of `src` and `xs` left out). Put the encoder in evaluation mode, so that batch norm treats the two strips independently, and compute `memory` for both. Predict, then report, the largest absolute change in the forward half and in the backward half of the states of column 1 and of column 21, and list the columns whose forward half changes at all. Explain the pattern from the directions in which the two GRUs read and from the receptive field you found in Q1.1.

In [ ]:
# YOUR CODE HERE

*Answer:*

## 3. Two readers

Both readers have the same three methods, which the supplied training and generation code in Section 4 calls:

- `encode(images)` returns $s_0$, of shape $(B, 64)$, and `memory`, whatever the decoder keeps from the image for the steps that follow;
- `step(y_prev, s_prev, memory)` runs one decoder step from the previous symbol, of shape $(B,)$, and the previous state, and returns the logits $(B, 13)$, the new state $(B, 64)$, and the attention weights $(B, 21)$, or `None` for a reader without attention;
- `forward(images, dec_in)` runs the decoder under teacher forcing over every position of `dec_in` and returns logits of shape $(B, T, 13)$.

The **fixed-context reader** is supplied. Its decoder is Lab 7's: $s_t = \operatorname{GRU}(E[y_{t-1}], s_{t-1})$ and $\ell_t = W_{\text{out}} s_t + b_{\text{out}}$, with an embedding width of 32, starting from the $s_0$ of the column encoder. Its `encode` returns `None` as the memory. The column states are computed, but they reach the decoder only through $s_0$, and everything the decoder learns about this particular strip must pass through those 64 numbers.

In [ ]:
class FixedReader(nn.Module):
    """The column encoder's s_0, then Lab 7's GRU decoder. The decoder never
    reads the column states: encode() returns None as the memory."""

    def __init__(self, hidden=64, emb=32):
        super().__init__()
        self.encoder = ColumnEncoder(hidden)
        self.emb = nn.Embedding(VOCAB, emb)
        self.gru = nn.GRU(emb, hidden, batch_first=True)
        self.out = nn.Linear(hidden, VOCAB)

    def encode(self, images):
        memory, s0 = self.encoder(images)
        return s0, None

    def step(self, y_prev, s_prev, memory):
        o, s = self.gru(self.emb(y_prev).unsqueeze(1), s_prev.unsqueeze(0))
        return self.out(o[:, 0]), s[0], None

    def forward(self, images, dec_in):
        s0, _ = self.encode(images)
        o, _ = self.gru(self.emb(dec_in), s0.unsqueeze(0))
        return self.out(o)

The **attention reader** keeps the column states and reads them again at every output step with scaled dot-product attention. The preceding decoder state supplies the query and each column state supplies a key and a value,

$$
q_t=W_Q\,s_{t-1},\qquad k_j=W_K\,h_j,\qquad v_j=W_V\,h_j,
$$

with $d_k = 64$ for the query and the keys and $d_v = 128$ for the values, and no biases. These are written for one vector at a time, as column vectors, which is how Lab 7 wrote $W_{\text{init}}$. The chapter stacks vectors as the rows of a matrix and writes $Q = X_q W_Q$, so its $W_Q$ is the transpose of the one here: $d_q \times d_k$ in the chapter, $d_k \times d_q$ here. `nn.Linear(64, 64, bias=False)` stores its weight in the shape used here, (out, in), and applied to a batch of row vectors it computes $X W^\top$, so the two conventions give the same numbers. The scores are normalised by a softmax over the 21 columns, and the context is the weighted sum of the values:

$$
e_{tj}=\frac{q_t\cdot k_j}{\sqrt{d_k}},\qquad
\alpha_{tj}=\frac{\exp(e_{tj})}{\sum_{i=1}^{21}\exp(e_{ti})},\qquad
c_t=\sum_{j=1}^{21}\alpha_{tj}\,v_j.
$$

The decoder then takes the context in two places, as part of the GRU's input and as part of the output layer's input:

$$
s_t=\operatorname{GRU}\bigl([E[y_{t-1}];\,c_t],\,s_{t-1}\bigr),\qquad
\ell_t=W_{\text{out}}[s_t;\,c_t]+b_{\text{out}}.
$$

The keys and values depend only on the image, so they are computed once per strip. The same $W_Q$, $W_K$, $W_V$ and decoder parameters are used at every step; what changes from step to step is the query, and so the weights.

**Q3.1.** Write `attend(q, K, V)`, scaled dot-product attention for one query per row of a batch: `q` has shape $(B, d_k)$, `K` $(B, N, d_k)$ and `V` $(B, N, d_v)$, and it returns the context $(B, d_v)$ and the weights $(B, N)$. Do not call `F.scaled_dot_product_attention` inside it. Then check it in three ways.

1. Use the chapter's worked example from the attention mixer, a single query $q = (1.2, 0.6)$, keys $(1.4, 0.9)$, $(-1.1, 1.3)$ and $(-0.8, -1.4)$, and values $(0.86, 0.20, 0.18)$, $(0.22, 0.62, 0.28)$ and $(0.48, 0.30, 0.78)$, and compare your weights and output with the chapter's.
2. Use a zero query. What must the weights be, whatever the keys?
3. Use random tensors of the sizes this lab uses ($B = 5$, $N = 21$, $d_k = 64$, $d_v = 128$, after `torch.manual_seed(0)`): check that every row of weights sums to one, and report the largest difference between your context and `F.scaled_dot_product_attention` given the query as a sequence of length one.

In [ ]:
def attend(q, K, V):
    # YOUR CODE HERE
    ...

In [ ]:
# YOUR CODE HERE

*Answer:*

**Q3.2.** Implement `AttentionReader(hidden=64, emb=32, d_k=64, d_v=128)` with the three methods above. It needs a `ColumnEncoder(hidden)` named `encoder`, an embedding table $E$, the projections $W_Q$, $W_K$ and $W_V$ as `nn.Linear` layers without biases, an `nn.GRUCell` whose input is the embedding and the context joined, and the output layer $W_{\text{out}}$ on the state and the context joined. `encode` returns $s_0$ and, as the memory, the keys and values of the 21 columns. `step` computes the query from `s_prev`, calls your `attend`, updates the state and returns the logits, the new state and the weights. `forward` calls `encode` once and then `step` at each position of `dec_in`.

In [ ]:
class AttentionReader(nn.Module):
    # YOUR CODE HERE
    ...

**Q3.3.** Before you count them: which parameters does the attention reader have that the fixed-context reader does not, and which of the fixed-context reader's layers become larger, and by how much? Compute the difference by hand, then print the parameter count of each part of both readers, the two totals and their ratio. Then the ritual from Lab 2: build one batch of 32 training strips, `StripDataset(train_imgs, train_labels, 32, range(1, 9), seed=999)`, call `seed_everything(0)`, and drive a fresh `AttentionReader` to memorise it with Adam at $10^{-3}$ for 300 steps under teacher forcing with `masked_loss`. Log it to W&B under a name that makes clear it is the ritual, print the loss at step 1 and every 50 steps, and report the first step at which it falls below 0.01.

In [ ]:
# YOUR CODE HERE

*Answer:*

## 4. Train and compare

The generation and training loops are supplied, because the recipe is Lab 7's and the comparison depends on both readers following it exactly. `generate` decodes greedily from `<sos>`, feeding back the highest-scoring symbol, and stops once every strip in the batch has written `<eos>` or after 14 steps, which is enough for a twelve-digit string and its `<eos>`. It returns the symbols, $(B, \text{steps})$, and for the attention reader the weights, $(B, \text{steps}, 21)$. `predict` runs it over a dataset and returns lists of predictions, targets and, for the attention reader, one array of weights per strip. Both put the model in evaluation mode, track no gradients, and return the model to the mode it was in.

The settings of the recipe are collected in one cell below. Change them only to check that your code runs end to end, and restore them before the runs you report. The one exception is `N_STRIPS`, discussed after the cell.

`train_reader` follows Lab 7's recipe on strips of one to eight digits: a fresh set of 60,000 training strips each epoch, batches of 128, Adam at $10^{-3}$ with cosine annealing stepped after every batch, twelve epochs, teacher forcing and `masked_loss`. It logs the mean training loss after every epoch, and the free-running exact-string accuracy and CER on `test_set` after the epochs listed in `EVAL_EPOCHS`, which saves evaluation time without changing the training. As in Lab 7 there is no separate validation split, so the per-epoch test numbers are for illustration only: keep the prescribed settings fixed and do not use the curves to choose them.

In [ ]:
EPOCHS = 12                      # training epochs for each reader
N_STRIPS = 60_000                # fresh training strips per epoch
BATCH = 128                      # training batch size
EVAL_EPOCHS = (1, 3, 6, 9, 12)   # epochs after which test_set is scored and logged
EVAL_BATCH = 500                 # batch size for generation

Training the two readers takes nearly all of a reference run's 38 minutes on a Colab T4. If you cannot spare that, you may lower `N_STRIPS` for the runs you report, provided you state the value you used in your answer to Q4.3; the W&B config records it as well. The cost of a smaller budget falls unevenly on the two readers. In three reference runs at 30,000 strips per epoch, half the prescribed number, training took about half as long and the attention reader still read about 96% of the strips of six to eight digits exactly, but the fixed-context reader fell from about 77% to about 40%. The fixed-context reader is still improving when training stops, so a smaller budget widens the gap between the readers, and part of the wider gap then reflects how much data each reader needs rather than what attention adds. If you lower `N_STRIPS`, your comparison in Section 4 has to say so.

In [ ]:
@torch.no_grad()
def generate(model, images, max_len=14):
    """Greedy decoding from <sos>; stops once every row has written <eos>.

    Returns the symbols (B, steps) and the attention weights (B, steps, 21),
    or None for a reader without attention.
    """
    was_training = model.training
    model.eval()
    s, memory = model.encode(images)
    y = torch.full((images.shape[0],), SOS, dtype=torch.long, device=images.device)
    out, weights = [], []
    for _ in range(max_len):
        logits, s, alpha = model.step(y, s, memory)
        y = logits.argmax(-1)
        out.append(y)
        if alpha is not None:
            weights.append(alpha)
        if (torch.stack(out, 1) == EOS).any(1).all():
            break
    model.train(was_training)
    return torch.stack(out, 1), (torch.stack(weights, 1) if weights else None)


@torch.no_grad()
def predict(model, dataset, batch_size=EVAL_BATCH):
    """Free-running predictions, targets and (attention only) weights, as lists."""
    preds, targets, weights = [], [], []
    for images, _, tgt in DataLoader(dataset, batch_size=batch_size, collate_fn=collate):
        p, a = generate(model, images.to(DEV))
        preds += p.tolist()
        targets += tgt.tolist()
        if a is not None:
            weights += list(a.cpu().numpy())
    return preds, targets, (weights if weights else None)


def train_reader(make_model, seed, name, lengths=range(1, 9)):
    """Lab 7's recipe on strips of `lengths`, logged to W&B. Returns the model
    and a list of per-epoch rows."""
    seed_everything(seed)
    model = make_model().to(DEV)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    steps_per_epoch = N_STRIPS // BATCH                       # 468
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS * steps_per_epoch)
    g = torch.Generator().manual_seed(seed)
    config = dict(reader=type(model).__name__, hidden=64, emb=32, bigru_units=64,
                  train_lengths=list(lengths), strips_per_epoch=N_STRIPS,
                  optimiser="Adam", lr=1e-3, schedule="cosine, per step",
                  batch_size=BATCH, epochs=EPOCHS, eval_epochs=list(EVAL_EPOCHS),
                  params=sum(p.numel() for p in model.parameters()), seed=seed)
    if isinstance(model, AttentionReader):
        config.update(attention="scaled dot-product", d_k=64, d_v=128)
    run = wandb.init(project="cv-lab8", name=name, config=config)
    history = []
    for epoch in range(1, EPOCHS + 1):
        strips = StripDataset(train_imgs, train_labels, N_STRIPS, lengths,
                              seed=1000 + epoch)
        loader = DataLoader(strips, batch_size=BATCH, shuffle=True, collate_fn=collate,
                            generator=g, drop_last=True)
        model.train()
        total = 0.0
        for images, dec_in, tgt in loader:
            images, dec_in, tgt = images.to(DEV), dec_in.to(DEV), tgt.to(DEV)
            loss = masked_loss(model(images, dec_in), tgt)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            total += loss.item()
        row = {"epoch": epoch, "train/loss": total / len(loader)}
        msg = f"epoch {epoch:2d}  train loss {row['train/loss']:.4f}"
        if epoch in EVAL_EPOCHS:
            preds, targets, _ = predict(model, test_set)
            row.update({"test/exact": exact_string_accuracy(preds, targets),
                        "test/cer": cer(preds, targets)})
            msg += f"  test exact {row['test/exact']:.4f}  CER {row['test/cer']:.4f}"
        wandb.log(row, step=epoch)
        history.append(row)
        print(msg)
    run.finish()
    return model, history

**Q4.1.** Before you train them: both readers train on the same strips of one to eight digits, with the same encoder and the same recipe. Which reader do you expect to lose more exact-string accuracy as the strips lengthen from one digit to eight, and why? Commit to an answer here; Q4.3 checks it.

*Answer:*

**Q4.2.** Train both readers with seed 0, the fixed-context reader first: `train_reader(FixedReader, 0, ...)` and `train_reader(AttentionReader, 0, ...)`, each with a meaningful run name. Plot the training loss against epoch and the test exact-string accuracy against the epochs at which it was measured, with both readers on each plot.

In [ ]:
# YOUR CODE HERE

**Q4.3.** Evaluate both readers on `test_set`. Report, for each reader, its parameter count and the exact-string accuracy and CER over lengths one to five together and over six to eight together, as a table, and plot exact-string accuracy and CER against length for both readers. Was your prediction in Q4.1 right? Describe where the two readers differ and where they do not. The attention reader has more parameters, so say what this comparison can attribute to attention and what it cannot.

In [ ]:
# YOUR CODE HERE

*Answer:*

## 5. Inspecting the alignments

`show_attention` reads one strip with the attention reader and draws two panels that share a horizontal axis: the strip with each digit's box outlined and labelled, and beneath it the attention weights, one row per output step and one cell per column of $F$, 16 pixels wide. All the rows of one strip share a colour scale, from zero to the largest weight in any of its rows, so a pale row is a row whose weight is spread out, not a row that has been rescaled. The row labels are the symbols the reader wrote.

In [ ]:
def show_attention(model, dataset, i, title=""):
    """Read strip i with an attention reader; draw the strip and its weights."""
    img, _ = dataset[i]
    pred, alpha = generate(model, img.unsqueeze(0).to(DEV))
    pred, alpha = pred[0].tolist(), alpha[0].cpu().numpy()
    steps = next((k + 1 for k, y in enumerate(pred) if y == EOS), len(pred))
    pred, alpha = pred[:steps], alpha[:steps]
    width = img.shape[-1]
    fig, (top, bottom) = plt.subplots(
        2, 1, figsize=(9, 1.6 + 0.3 * steps), sharex=True, layout="constrained",
        gridspec_kw=dict(height_ratios=[1.3, 0.3 * steps]))
    top.imshow(img[0], cmap="gray", extent=(0, width, H, 0))
    for d, x in zip(dataset.digits[i], dataset.xs[i]):
        top.add_patch(plt.Rectangle((x, 0.5), 28, H - 1, fill=False, ec="tab:orange", lw=1))
        top.text(x + 14, -1, str(d), ha="center", va="bottom", fontsize=8, color="tab:orange")
    top.set_yticks([])
    top.set_title(f"{title}target {''.join(map(str, dataset.digits[i]))}   "
                  f"read {''.join(map(str, clean(pred)))}", loc="left", fontsize=9, pad=12)
    im = bottom.imshow(alpha, cmap="viridis", vmin=0, vmax=alpha.max(), aspect="auto",
                       interpolation="nearest", extent=(0, width, steps - 0.5, -0.5))
    for x in dataset.xs[i]:
        bottom.axvline(x, color="white", lw=0.6, alpha=0.6)
    bottom.set_yticks(range(steps), [SYMBOLS[y] for y in pred], fontsize=8)
    bottom.set_ylabel("written")
    bottom.set_xlabel("strip position (pixels); one attention cell per 16 pixels")
    fig.colorbar(im, ax=[top, bottom], fraction=0.025, pad=0.01, label="weight")
    plt.show()

**Q5.1.** Find a strip of seven or eight digits in `test_set` that the attention reader reads exactly, and a strip of six to eight digits that it reads wrongly, and display both with `show_attention`. For the correct reading, describe where the weight falls at each output step, including the step that writes `<eos>`, and how large the largest weight in each row is. For the failure, find the first wrong symbol and describe the weights at that step and the steps after it. Say what the maps show and what they leave open: which column state received the largest weight in a context, compared with which part of the image the written symbol depended on.

In [ ]:
# YOUR CODE HERE

*Answer:*

**Q5.2.** Measure the alignment over the whole of `test_set`. `generate` keeps decoding a strip until every strip in its batch has finished, so a prediction can contain symbols after its own `<eos>`; discard them, since the reader has stopped. For each strip, list the digits written before the first `<eos>`, together with the generation step at which each was written (skip any `<sos>` or `<pad>`, as `clean` does). Then, for each target position $k$ (counting from 0):

- if the reader wrote fewer than $k + 1$ digits, count position $k$ as **missing** and do not score it;
- otherwise take the weights at the step that wrote its $k$-th digit, and the column $c$ (counting from 0) where they are largest. Column $c$ corresponds to strip positions $16c$ to $16c + 15$, so take its centre to be pixel $16c + 8$, and count the position as **aligned** if that centre lies inside the box of the target's $k$-th digit, from `xs[k]` to `xs[k] + 27`.

Report the number of missing positions, the share of aligned positions among the digits read correctly (the $k$-th digit written equals the target's $k$-th digit) and among those read wrongly, and the median of the largest weight over the correctly read digits. What does a high share establish about how the reader works, and what does it not?

In [ ]:
# YOUR CODE HERE

*Answer:*

**Q5.3.** An attention weight is part of the computation. A claim that a prediction depends on part of the image needs an intervention and a measured change in behaviour. Take the first 1,000 strips of six to eight digits in `test_set` that both readers read exactly, so that the two readers are tested on the same strips. In each, choose one position $k$ at random with `np.random.default_rng(5)`, replace that digit's source image by a random MNIST test image of a different class, redraw the strip with `draw_strip` at the same box positions, and give the same new strips to both readers. Report three proportions for each reader.

1. The proportion of strips read exactly as the new string.
2. The proportion whose reading has a $k$-th digit equal to the new digit, whatever happens elsewhere.
3. The proportion whose reading changed in length or in any digit other than the $k$-th.

What does this experiment establish that Q5.2 does not, and what does it leave open? This experiment does not require attention weights.

In [ ]:
# YOUR CODE HERE

*Answer:*

## 6. Beyond the training lengths

Neither reader has seen a strip of more than eight digits. Both can process one: the canvas holds twelve, the encoder produces 21 column states whatever the strip holds, and `generate` allows 14 steps. Being able to accept a longer input is not the same as reading it correctly, and this section measures the difference.

**Q6.1.** Predict, before you run it, how each reader will do on strips of nine to twelve digits, and whether attention will help there. Then evaluate both readers on `long_set`, and report, for each reader and each length from nine to twelve, the exact-string accuracy, the CER and how often each output length occurs. Classify each strip as Lab 7 did, by comparing cleaned lengths: correct, *missed* digits (shorter than the target), *extra* digits (longer) or *incorrect* digits (the same length but different). Also report the position of the first wrong digit, counting from the first digit as position 0, as a count per position. Which reader does better, by which measure, and why can exact-string accuracy and CER rank the readers differently at these lengths?

In [ ]:
# YOUR CODE HERE

*Answer:*

**Q6.2.** Display the attention maps of three strips from `long_set` that the attention reader reads wrongly, including at least one of twelve digits, and compute the Q5.2 alignment share on `long_set`. Describe where the weights go at the steps before the first error and at and after it, and compare the length of what the reader wrote with the length of the target. Propose an explanation for the failure, and say what evidence would test it. Attention weights show how the column states were mixed into each context; they do not on their own show why the decoder wrote what it wrote, so state which parts of your explanation the maps support and which they do not.

In [ ]:
# YOUR CODE HERE

*Answer:*

## 7. Optional explorations

These are not marked, and neither is needed to complete the experiment above.

- The caption-attention widget in Chapter 8 shows recorded attention for a model generating words about a photograph. Compare its maps with this reader's: how concentrated is the weight in each, and what does a caption word have that corresponds to a digit's box, if anything?
- Use the chapter's position-code explorer to test the same-offset relationship, or implement sinusoidal position codes from the chapter's formula and plot the dot product between the codes of positions $p$ and $q$ against $p - q$.

## 8. W&B runs

Paste links to your W&B runs below, three at minimum, each with a meaningful name and a config that records the following.

- **The one-batch ritual:** reader, state and embedding widths, $d_k$ and $d_v$, number of strips in the batch, number of steps, optimiser, learning rate, parameter count and seed.
- **The two readers:** the configs `train_reader` writes, which carry the reader, the widths, the training lengths, strips per epoch, optimiser, learning rate and schedule, batch size, epochs, parameter count and seed. Each logs the training loss per epoch, and test exact-string accuracy and CER at the epochs in `EVAL_EPOCHS`.

*W&B run links:*

## 9. Before you submit

- [ ] **Runtime ▸ Restart session and run all** on a GPU runtime, then read every output. The full re-run downloads MNIST, runs the ritual and trains two readers. A reference run took about 38 minutes on a Colab T4; budget fifty, and longer on a slower GPU.
- [ ] Fill in the group members; check that every member can explain every cell.
- [ ] Complete every *Answer:* cell, and paste the W&B links into Section 8.
- [ ] **File ▸ Download ▸ Download .ipynb**; one member submits it on Moodle by the deadline shown there.